# 02 - Point-supervised segmentation on LoveDA with partial focal cross entropy

## 0. Setup

Clones or pulls the repo to get the split files in `splits/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ABDULAH-ab/point-supervised-remote-sensing-segmentation.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/tmp/point-supervised-remote-sensing-segmentation')

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass  # no secret attached: fine for a public repo

auth_url = REPO_URL.replace('https://', 'https://' + token + '@') if token else REPO_URL


def git(*args):
    result = subprocess.run(['git', *args], capture_output=True, text=True)
    output = result.stdout + result.stderr
    if token:
        output = output.replace(token, '***')  # never print the token
    print(output.strip())
    if result.returncode != 0:
        raise RuntimeError('git ' + args[0] + ' failed - see the message above')


if (REPO_DIR / '.git').exists():
    git('-C', str(REPO_DIR), 'pull', '--ff-only', auth_url, BRANCH)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git('clone', '--branch', BRANCH, auth_url, str(REPO_DIR))
    git('-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL)  # do not keep the token in the clone

git('-C', str(REPO_DIR), 'log', '-1', '--oneline')

SPLITS_DIR = REPO_DIR / 'splits'
assert (SPLITS_DIR / 'train.txt').exists(), 'splits/ is not in the repo yet - commit and push the three txt files'

In [ ]:
import hashlib
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import segmentation_models_pytorch as smp
except ImportError:
    !pip install -q segmentation-models-pytorch
    import segmentation_models_pytorch as smp

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

## 1. Configuration

Every setting of a run lives in one dictionary, so an experiment is the default config with one or two values changed.

In [ ]:
# TODO: fill in and adjust
NUM_CLASSES = 7
IGNORE_INDEX = 255
CLASS_NAMES = ['background', 'building', 'road', 'water', 'barren', 'forest', 'agriculture']

DEFAULT_CONFIG = {
    'name': 'baseline',          # used for the results file name
    'seed': 42,
    'points_per_image': 20,      # None means full masks (the upper bound run)
    'sampling': 'uniform',       # 'uniform' or 'class_balanced'
    'gamma': 2.0,                # focal strength; 0 is plain partial cross entropy
    'crop_size': 512,
    'batch_size': 8,
    'epochs': 20,
    'lr': 1e-3,
    'encoder': 'resnet34',
    'pretrained': True,
}

## 2. Partial focal cross entropy loss

$$\mathrm{pfCE} = \frac{\sum \mathrm{FocalLoss}(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

The per-pixel focal loss is kept only at labelled pixels and averaged over the number of labelled pixels.

In [ ]:
class PartialFocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        mask = target != self.ignore_index
        ce = F.cross_entropy(logits, target, ignore_index = self.ignore_index, reduction = 'none')
        p_t = torch.exp(-ce)
        focal = (1 - p_t) ** self.gamma * ce

        return (focal * mask).sum() / mask.sum().clamp(min=1)

### Tests for the loss

These run on small random tensors and need no data or GPU.

In [ ]:
torch.manual_seed(0)

logits = torch.randn(2, NUM_CLASSES, 16, 16, requires_grad=True)
target = torch.full((2, 16, 16), IGNORE_INDEX, dtype=torch.long)
target[0, 2, 3] = 1
target[0, 10, 12] = 4
target[1, 5, 5] = 0
target[1, 14, 8] = 6

# Test 1: gamma=0 equals F.cross_entropy
loss_g0 = PartialFocalCELoss(gamma=0.0, ignore_index=IGNORE_INDEX)(logits, target)
ref_ce = F.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)
assert torch.allclose(loss_g0, ref_ce)

# Test 2: adding noise at unlabelled pixels does not change the loss
noisy_logits = logits.detach().clone()
unlabelled = (target == IGNORE_INDEX).unsqueeze(1)
noisy_logits = noisy_logits + torch.randn_like(noisy_logits) * unlabelled
loss_g2 = PartialFocalCELoss(gamma=2.0, ignore_index=IGNORE_INDEX)(logits, target)
loss_noisy = PartialFocalCELoss(gamma=2.0, ignore_index=IGNORE_INDEX)(noisy_logits, target)
assert torch.allclose(loss_g2, loss_noisy)

# Test 3: gradients are 0 at unlabelled pixels and non-zero at some labelled pixel
loss_g2.backward()
grad_bhwc = logits.grad.permute(0, 2, 3, 1)
assert (grad_bhwc[target == IGNORE_INDEX] == 0).all()
assert (grad_bhwc[target != IGNORE_INDEX].abs() > 0).any()

# Test 4: all-255 target gives 0, not NaN
all_ignored = torch.full((2, 16, 16), IGNORE_INDEX, dtype=torch.long)
loss_empty = PartialFocalCELoss(gamma=2.0, ignore_index=IGNORE_INDEX)(logits, all_ignored)
assert not torch.isnan(loss_empty) and loss_empty.item() == 0.0

# Test 5: gamma=2 gives a smaller loss than gamma=0 on the same inputs
assert loss_g2 < loss_g0

print('all loss tests passed')

## 3. Simulating point labels

Turns a full mask into a sparse one: a fixed number of pixels keep their label and all others become 255.

The points of an image are fixed by a seed built from the run seed and the file name, so the same image always gets the same points in every epoch.

In [ ]:
def sample_points(mask, n_points, strategy, rng):
    """Keep n_points labelled pixels in mask (0..6) and set all others to IGNORE_INDEX."""
    valid_ys, valid_xs = np.where(mask != IGNORE_INDEX)
    n_valid = len(valid_ys)
    out = np.full_like(mask, IGNORE_INDEX)
    if n_valid == 0 or n_points <= 0:
        return out
    if n_valid <= n_points:
        out[valid_ys, valid_xs] = mask[valid_ys, valid_xs]
        return out

    if strategy == 'uniform':
        chosen = rng.choice(n_valid, size=n_points, replace=False)
        ys, xs = valid_ys[chosen], valid_xs[chosen]
        out[ys, xs] = mask[ys, xs]
    elif strategy == 'class_balanced':
        classes = [c for c in np.unique(mask) if c != IGNORE_INDEX]
        rng.shuffle(classes)
        base, rem = divmod(n_points, len(classes))
        picked_mask = np.zeros(n_valid, dtype=bool)
        valid_labels = mask[valid_ys, valid_xs]
        for i, cls in enumerate(classes):
            quota = base + (1 if i < rem else 0)
            cls_idx = np.where(valid_labels == cls)[0]
            take = min(quota, len(cls_idx))
            if take > 0:
                chosen = rng.choice(cls_idx, size=take, replace=False)
                picked_mask[chosen] = True
        shortfall = n_points - int(picked_mask.sum())
        if shortfall > 0:
            remaining = np.where(~picked_mask)[0]
            extra = rng.choice(remaining, size=shortfall, replace=False)
            picked_mask[extra] = True
        ys, xs = valid_ys[picked_mask], valid_xs[picked_mask]
        out[ys, xs] = mask[ys, xs]
    else:
        raise ValueError(f'Unknown sampling strategy: {strategy}')
    return out


# Check determinism, point count, and visualise on a synthetic mask
synth = np.zeros((64, 64), dtype=np.uint8)
synth[:32, :32] = 1
synth[32:, 32:] = 2
synth[:10, :10] = IGNORE_INDEX

p1 = sample_points(synth, 20, 'uniform', np.random.default_rng(123))
p2 = sample_points(synth, 20, 'uniform', np.random.default_rng(123))
assert np.array_equal(p1, p2)
assert int((p1 != IGNORE_INDEX).sum()) == 20

p_bal = sample_points(synth, 20, 'class_balanced', np.random.default_rng(123))
assert int((p_bal != IGNORE_INDEX).sum()) == 20

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(synth, vmin=0, vmax=6, cmap='tab10', interpolation='nearest')
axes[0].set_title('Synthetic mask')
py, px = np.where(p_bal != IGNORE_INDEX)
axes[1].imshow(synth, vmin=0, vmax=6, cmap='tab10', interpolation='nearest', alpha=0.4)
axes[1].scatter(px, py, c='red', s=25, marker='x')
axes[1].set_title('20 class-balanced points')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Dataset and augmentations

Label remapping: mask value 0 (no-data) becomes 255, values 1-7 become classes 0-6.

In [ ]:
INPUT = Path('/kaggle/input')
DIRS = {}
for root, dirs, _ in os.walk(INPUT):
    if 'images_png' not in dirs:
        continue
    root = Path(root)
    scene = root.name
    split = root.parent.name
    if scene in ('Urban', 'Rural') and split in ('Train', 'Val'):
        assert (split, scene) not in DIRS, f'Found two copies of {split}/{scene}'
        DIRS[(split, scene)] = root

KEYS = [('Train', 'Urban'), ('Train', 'Rural'), ('Val', 'Urban'), ('Val', 'Rural')]
missing = [k for k in KEYS if k not in DIRS]
assert not missing, f'Folders not found: {missing}'


def read_split(path):
    return [line.strip() for line in Path(path).read_text().splitlines() if line.strip()]


train_entries = read_split(SPLITS_DIR / 'train.txt')
val_entries = read_split(SPLITS_DIR / 'val.txt')
test_entries = read_split(SPLITS_DIR / 'test.txt')
print(f'Split sizes -> train: {len(train_entries)}, val: {len(val_entries)}, test: {len(test_entries)}')

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
NUM_WORKERS = min(4, os.cpu_count() or 1)


def _image_rng(run_seed, entry):
    digest = hashlib.sha256(f'{run_seed}:{entry}'.encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], 'little'))


class LoveDAPoints(torch.utils.data.Dataset):
    def __init__(self, entries, dirs, config, train=True):
        self.entries = entries
        self.dirs = dirs
        self.config = config
        self.train = train

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        split, scene, fname = entry.split('/')
        folder = self.dirs[(split, scene)]

        img = np.array(Image.open(folder / 'images_png' / fname).convert('RGB'))
        raw_mask = np.array(Image.open(folder / 'masks_png' / fname))

        mask = np.full_like(raw_mask, IGNORE_INDEX)
        valid = raw_mask > 0
        mask[valid] = raw_mask[valid] - 1

        if self.train and self.config['points_per_image'] is not None:
            rng = _image_rng(self.config['seed'], entry)
            mask = sample_points(mask, self.config['points_per_image'], self.config['sampling'], rng)

        if self.train:
            cs = self.config['crop_size']
            h, w = mask.shape
            if h > cs and w > cs:
                top = np.random.randint(0, h - cs + 1)
                left = np.random.randint(0, w - cs + 1)
                img = img[top:top + cs, left:left + cs]
                mask = mask[top:top + cs, left:left + cs]
            if np.random.random() < 0.5:
                img = np.fliplr(img)
                mask = np.fliplr(mask)
            if np.random.random() < 0.5:
                img = np.flipud(img)
                mask = np.flipud(mask)

        img = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        mask = np.ascontiguousarray(mask)
        return torch.from_numpy(img), torch.from_numpy(mask).long()


def _eval_loader(entries, config):
    # Full masks, no augmentation, fixed order
    ds = LoveDAPoints(entries, DIRS, config, train=False)
    return torch.utils.data.DataLoader(
        ds, batch_size=config['batch_size'], shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda')
    )


def make_loaders(config):
    tr_entries = train_entries[:config['max_train_tiles']] if 'max_train_tiles' in config else train_entries
    va_entries = val_entries[:config['max_val_tiles']] if 'max_val_tiles' in config else val_entries
    train_ds = LoveDAPoints(tr_entries, DIRS, config, train=True)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'), drop_last=True
    )
    return train_loader, _eval_loader(va_entries, config)


def make_test_loader(config):
    # 'max_val_tiles' also shortens the test set, so the smoke test stays quick
    te_entries = test_entries[:config['max_val_tiles']] if 'max_val_tiles' in config else test_entries
    return _eval_loader(te_entries, config)


# Check one training sample
sample_ds = LoveDAPoints(train_entries[:4], DIRS, DEFAULT_CONFIG, train=True)
img_t, mask_t = sample_ds[0]
print('image:', img_t.shape, img_t.dtype, '| mask:', mask_t.shape, mask_t.dtype, 'unique:', torch.unique(mask_t).tolist())

vis_img = np.clip(img_t.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
py, px = np.where(mask_t.numpy() != IGNORE_INDEX)
plt.figure(figsize=(5, 5))
plt.imshow(vis_img)
plt.scatter(px, py, c=mask_t.numpy()[py, px], cmap='tab10', vmin=0, vmax=6, s=30, edgecolors='white')
plt.title('Training crop with point labels')
plt.axis('off')
plt.show()

## 5. Model

In [ ]:
def build_model(config):
    model = smp.Unet(
        encoder_name=config['encoder'],
        encoder_weights='imagenet' if config['pretrained'] else None,
        in_channels=3,
        classes=NUM_CLASSES,
    )
    return model.to(DEVICE)


_m = build_model(DEFAULT_CONFIG)
_x = torch.randn(2, 3, DEFAULT_CONFIG['crop_size'], DEFAULT_CONFIG['crop_size'], device=DEVICE)
with torch.no_grad():
    _y = _m(_x)
print('model output shape:', _y.shape)
del _m, _x, _y

## 6. Metric: IoU from a confusion matrix

Accumulate one confusion matrix over the whole evaluation set, then compute per-class IoU and their mean. Pixels with label 255 are skipped.

In [ ]:
def update_confusion(conf, pred, target):
    if torch.is_tensor(pred):
        pred = pred.detach().cpu().numpy()
    if torch.is_tensor(target):
        target = target.detach().cpu().numpy()
    valid = (target != IGNORE_INDEX) & (target >= 0) & (target < NUM_CLASSES)
    idx = target[valid].astype(np.int64) * NUM_CLASSES + pred[valid].astype(np.int64)
    conf += np.bincount(idx, minlength=NUM_CLASSES * NUM_CLASSES).reshape(NUM_CLASSES, NUM_CLASSES)


def iou_from_confusion(conf):
    tp = np.diag(conf)
    fp = conf.sum(axis=0) - tp
    fn = conf.sum(axis=1) - tp
    denom = tp + fp + fn
    per_class = np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)
    miou = float(np.nanmean(per_class))
    return per_class, miou


# Hand-made check: 2 classes present (0 and 1) + ignored pixels (255)
# Class 0: TP=2, FP=1, FN=0 -> IoU = 2/3
# Class 1: TP=1, FP=0, FN=1 -> IoU = 1/2
# mIoU = (2/3 + 1/2) / 2 = 7/12
_conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
_pred = np.array([[0, 0, 0, 1, 1]])
_targ = np.array([[0, 0, 1, 1, IGNORE_INDEX]])
update_confusion(_conf, _pred, _targ)
_pc, _miou = iou_from_confusion(_conf)
assert np.isclose(_pc[0], 2 / 3) and np.isclose(_pc[1], 0.5) and np.isclose(_miou, 7 / 12)
print('confusion matrix IoU check passed')

## 7. Training and evaluation

In [ ]:
RESULTS_DIR = Path('/kaggle/working/results')
CKPT_DIR = Path('/kaggle/working/checkpoints')
USE_AMP = DEVICE == 'cuda'  # mixed precision: faster and lighter on the GPU, off on CPU


def make_scaler():
    try:
        return torch.amp.GradScaler(DEVICE, enabled=USE_AMP)
    except (AttributeError, TypeError):  # older PyTorch
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def evaluate(model, loader):
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                logits = model(images)
            update_confusion(conf, logits.argmax(dim=1), masks)
    return iou_from_confusion(conf)


def iou_dict(per_class):
    return {name: (None if np.isnan(val) else float(val)) for name, val in zip(CLASS_NAMES, per_class)}


def run_experiment(config):
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    res_path = RESULTS_DIR / f"{config['name']}.json"
    ckpt_path = CKPT_DIR / f"{config['name']}.pt"
    if res_path.exists():
        print(f"Skipping {config['name']} (already exists at {res_path})")
        return json.loads(res_path.read_text())

    seed_everything(config['seed'])
    train_loader, val_loader = make_loaders(config)
    model = build_model(config)
    criterion = PartialFocalCELoss(gamma=config['gamma'], ignore_index=IGNORE_INDEX)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'])
    scaler = make_scaler()

    train_losses, val_mious = [], []
    best_miou = -1.0
    best_per_class = None
    t0 = time.time()

    for epoch in range(1, config['epochs'] + 1):
        t_epoch = time.time()
        model.train()
        total_loss = 0.0
        for images, masks in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                logits = model(images)
            loss = criterion(logits.float(), masks)  # the loss itself is computed in full precision
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        avg_loss = total_loss / max(len(train_loader), 1)
        per_class, val_miou = evaluate(model, val_loader)
        train_losses.append(avg_loss)
        val_mious.append(val_miou)

        if val_miou > best_miou:
            best_miou = val_miou
            best_per_class = per_class
            torch.save(model.state_dict(), ckpt_path)

        print(f"[{config['name']}] epoch {epoch:02d}/{config['epochs']} | loss {avg_loss:.4f} "
              f"| val mIoU {val_miou:.4f} | {time.time() - t_epoch:.0f}s")

    elapsed = time.time() - t0

    # Test score of the checkpoint chosen on validation. The test set plays no part in choosing it.
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_per_class, test_miou = evaluate(model, make_test_loader(config))
    print(f"[{config['name']}] best val mIoU {best_miou:.4f} | test mIoU {test_miou:.4f}")

    result = {
        'config': config,
        'train_loss': train_losses,
        'val_miou': val_mious,
        'best_miou': float(best_miou),
        'best_per_class_iou': iou_dict(best_per_class),
        'test_miou': float(test_miou),
        'test_per_class_iou': iou_dict(test_per_class),
        'train_time_sec': round(elapsed, 2),
    }
    res_path.write_text(json.dumps(result, indent=2))
    return result

## 8. Smoke test

A tiny run (few tiles, 1-2 epochs) to prove the whole pipeline works before spending GPU hours.

In [ ]:
smoke_file = RESULTS_DIR / '_smoke_test.json'
if smoke_file.exists():
    smoke_file.unlink()

smoke_cfg = {
    **DEFAULT_CONFIG,
    'name': '_smoke_test',
    'epochs': 2,
    'batch_size': 4,
    'points_per_image': 500,   # enough labelled pixels for a stable loss on a handful of tiles
    'max_train_tiles': 16,
    'max_val_tiles': 8,
}
smoke_res = run_experiment(smoke_cfg)
assert smoke_file.exists()
assert all(np.isfinite(smoke_res['train_loss'])), 'Loss is NaN or infinite'
assert 0.0 <= smoke_res['test_miou'] <= 1.0
print('Smoke test passed: the pipeline runs end to end. Losses per epoch:', [round(x, 4) for x in smoke_res['train_loss']])

## 9. Experiment A - number of points per image

**Purpose:** Measure how segmentation mIoU scales with the number of annotated points per image, and compare against full-mask supervision.

**Hypothesis:** Increasing points per image improves validation mIoU with diminishing returns, approaching the full-mask upper bound.

**Process:** Train with `points_per_image` in `[5, 20, 100, 500, None]` (keeping `gamma=2.0` and `sampling='uniform'` fixed).

In [ ]:
for n_pts in [5, 20, 100, 500, None]:
    run_name = f'pts_{n_pts}' if n_pts is not None else 'pts_full'
    cfg = {**DEFAULT_CONFIG, 'name': run_name, 'points_per_image': n_pts}
    run_experiment(cfg)

## 10. Experiment B - focal gamma

**Purpose:** Test whether focal weighting (`gamma > 0`) improves point-supervised segmentation over plain partial cross entropy (`gamma = 0`).

**Hypothesis:** Moderate focal weighting (`gamma = 1.0` or `2.0`) focuses learning on harder boundary/minority-class points and outperforms `gamma = 0.0`.

**Process:** Train with `gamma` in `[0.0, 1.0, 2.0]` at `points_per_image=20`, each with three seeds. The seed changes both the sampled points and the training randomness, so the spread across seeds shows how much of a difference is noise. The run with `gamma=2.0` and seed 42 is the `pts_20` run from Experiment A and is not trained twice.

In [ ]:
GAMMAS = [0.0, 1.0, 2.0]
GAMMA_SEEDS = [42, 43, 44]


def gamma_run_name(gamma, seed):
    # The default gamma with the default seed is the same run as pts_20 from Experiment A
    if gamma == DEFAULT_CONFIG['gamma'] and seed == DEFAULT_CONFIG['seed'] and DEFAULT_CONFIG['points_per_image'] == 20:
        return 'pts_20'
    return f'gamma_{gamma}_s{seed}'


# Seeds in the outer loop: after each seed there is one complete comparison across all gammas
for seed in GAMMA_SEEDS:
    for gamma in GAMMAS:
        cfg = {**DEFAULT_CONFIG, 'name': gamma_run_name(gamma, seed), 'gamma': gamma, 'seed': seed}
        run_experiment(cfg)

## 11. Results

Reads the saved results files, so this section can be re-run without retraining.

In [ ]:
all_results = {}
for p in sorted(RESULTS_DIR.glob('*.json')):
    if p.stem.startswith('_'):
        continue
    all_results[p.stem] = json.loads(p.read_text())

# Summary table: validation mIoU chose the checkpoint, test mIoU and per-class IoU are the reported numbers
header = f"{'Run':<16} {'val':>7} {'test':>7} | " + ' '.join(f'{c[:6]:>6}' for c in CLASS_NAMES)
print(header)
print('-' * len(header))
for name, res in all_results.items():
    cls_str = ' '.join(
        f"{res['test_per_class_iou'][c]:>6.3f}" if res['test_per_class_iou'][c] is not None else '   nan'
        for c in CLASS_NAMES
    )
    print(f"{name:<16} {res['best_miou']:>7.4f} {res['test_miou']:>7.4f} | {cls_str}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Experiment A: test mIoU against points per image
pts_x, pts_y = [], []
for n_pts in [5, 20, 100, 500]:
    k = f'pts_{n_pts}'
    if k in all_results:
        pts_x.append(n_pts)
        pts_y.append(all_results[k]['test_miou'])
if pts_x:
    axes[0].plot(pts_x, pts_y, marker='o', label='Point-supervised')
    axes[0].set_xscale('log')
if 'pts_full' in all_results:
    axes[0].axhline(all_results['pts_full']['test_miou'], color='k', linestyle='--', label='Full masks')
axes[0].set_xlabel('Points per 1024x1024 tile (log scale)')
axes[0].set_ylabel('Test mIoU')
axes[0].set_title('Experiment A: points per image')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Experiment B: mean and spread of test mIoU over seeds for each gamma
print()
print(f"{'gamma':<8} {'runs':>4} {'mean test mIoU':>15} {'std':>8}   per-seed values")
g_x, g_mean, g_std = [], [], []
for g in GAMMAS:
    scores = [all_results[gamma_run_name(g, s)]['test_miou'] for s in GAMMA_SEEDS if gamma_run_name(g, s) in all_results]
    if not scores:
        continue
    g_x.append(g)
    g_mean.append(float(np.mean(scores)))
    g_std.append(float(np.std(scores)))
    print(f"{g:<8} {len(scores):>4} {g_mean[-1]:>15.4f} {g_std[-1]:>8.4f}   " + ', '.join(f'{s:.4f}' for s in scores))
if g_x:
    axes[1].errorbar(g_x, g_mean, yerr=g_std, marker='s', capsize=4)
axes[1].set_xlabel('Focal gamma')
axes[1].set_ylabel('Test mIoU (mean and std over seeds)')
axes[1].set_title('Experiment B: focal gamma (20 points)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Predictions of the best point-supervised run (chosen by validation mIoU)
point_runs = {k: v for k, v in all_results.items() if k != 'pts_full'}
if point_runs:
    best_name = max(point_runs, key=lambda k: point_runs[k]['best_miou'])
    best_cfg = all_results[best_name]['config']
    model = build_model(best_cfg)
    model.load_state_dict(torch.load(CKPT_DIR / f'{best_name}.pt', map_location=DEVICE))
    model.eval()

    vis_ds = LoveDAPoints(test_entries[:3], DIRS, best_cfg, train=False)
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for row in range(len(vis_ds)):
        img_t, mask_t = vis_ds[row]
        with torch.no_grad():
            pred = model(img_t.unsqueeze(0).to(DEVICE)).argmax(dim=1).squeeze(0).cpu().numpy()
        vis_img = np.clip(img_t.numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN, 0, 1)
        axes[row, 0].imshow(vis_img)
        axes[row, 0].set_title('Image')
        axes[row, 1].imshow(mask_t.numpy(), vmin=0, vmax=6, cmap='tab10', interpolation='nearest')
        axes[row, 1].set_title('Ground truth')
        axes[row, 2].imshow(pred, vmin=0, vmax=6, cmap='tab10', interpolation='nearest')
        axes[row, 2].set_title(f'Prediction ({best_name})')
        for col in range(3):
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()

## 12. Conclusions

TODO: what the experiments showed, whether each hypothesis held, and the limitations.